In [1]:
# ==============================================================================
# DATA ANONYMIZATION AND AUTOMATED EXPLORATORY DATA ANALYSIS (EDA) PIPELINE
# ------------------------------------------------------------------------------
# Business Goal: Load, harmonize, anonymize, and profile multi-department 
# operational datasets while ensuring strict enterprise data privacy and 
# relational integrity for downstream Data Warehousing.
# ==============================================================================

# Pandas: Used for data manipulation, cleaning, and structured DataFrames
import pandas as pd
# NumPy: Used for numerical operations and handling missing values
import numpy as np
# ProfileReport (ydata-profiling): Generates automated HTML EDA summary reports
from data_profiling import ProfileReport
# Glob: Finds files matching a specific path or extension pattern (e.g., *.csv)
import glob
# Re (Regular Expressions): Used for text cleaning and string pattern matching
import re
# OS: Handles operating system tasks like file paths and extracting filenames
import os
# Pathlib (Path): Provides clean, cross-platform file path management
from pathlib import Path

In [2]:
# Step 1: Use glob to scan the raw data directory and List all CSV files
# The '../data_raw/*.csv' path tells Python to look one folder up, enter 'data_raw', and select every CSV file
folder = glob.glob("../data/raw_data/*.csv")
try:
    for file_path in folder:
        file_name = os.path.basename(file_path)
        df = pd.read_csv(file_path, nrows=1)
        print(f"file name: {file_name}")
        print(df.columns.tolist())
        print("_" * 60)

except Exception as e:
    print(f"Error on file {file_path}: {e}")

file name: Fabric_Stock_level.csv
['Fabric SKU', 'Fabric Name', 'Print', 'Color', 'Composition', 'Weave', 'Season', 'Drop (s)', 'Status', 'Channel', 'OH', 'Reserve', 'ATS']
____________________________________________________________
file name: Fabric_tracker_Daily_log.csv
['Date', 'Month', 'Year', 'Fabric SKU', 'Fabric Name', 'Status', 'Action Type', 'Qty Change', 'Unnamed: 8', 'Unnamed: 9', 'Stock tracking model (ATS) : Opening Stock + Δ - Reserve']
____________________________________________________________
file name: Finished_Clothes_Daily_Log.csv
['Date', 'Model', 'Fabric', 'Key2', 'Product Full Name', 'VPN', 'Size', 'Combo_VPN_Size', 'Height(Cm)', 'Combo_SKU_Custom', 'Product_SKU', 'Combo_Product_SKU', 'Stock Type', 'Quality_type', 'Storage_Ref', 'Action Type', 'Qty']
____________________________________________________________
file name: Finished_Clothes_Inventory.csv
['Appartment', 'Room', 'Box', 'Stock Category', 'Product_quality', 'Product SKU', 'Product Type', 'Product Mode

### I - *Standardization of Database Column Names*

In [3]:
# Function to clean and standardize headers for dataset

columns_to_drop = ["Drop (s)", "Status", "Channel", "OH", "Reserve", "Fabric Name"]
def clean_headers(df):
    """
    Cleans and standardizes the fabric stock dataframe headers,
    renames specific key columns, and removes unwanted empty columns.    
    """
    # Step 1: Safely drop unnecessary columns from fabric stock database 
    df.drop(columns=columns_to_drop, errors="ignore", inplace=True)
    
    # Step 2: Strip leading and trailing whitespaces from column headers
    df.columns = df.columns.str.strip()

    # Step 3 : Replace spaces with underscores and convert all headers to lowercase
    df.columns = df.columns.str.replace(" ", "_").str.lower()   

    return df

In [4]:
# Step 1: Load the "Fabric stock tracker" CSV file into a pandas DataFrame
fabric_stock_df = pd.read_csv("../data/raw_data/Fabric_Stock_level.csv")
# Step 2: Apply the standardized header cleaning function
fabric_stock_df = clean_headers(fabric_stock_df)
# Step 3: Print and inspect the cleaned column names list
print(f"dataset : 'fabric_stock_df'")
print(fabric_stock_df.columns.tolist())

dataset : 'fabric_stock_df'
['fabric_sku', 'print', 'color', 'composition', 'weave', 'season', 'ats']


In [5]:
# Step 1: Load the "Fabric stock tracker Daily log" CSV file into a pandas DataFrame
fabric_daily_log_df = pd.read_csv("../data/raw_data/Fabric_tracker_Daily_log.csv")
# Step 2 List of columns to delete 
columns_to_drop = ['Unnamed: 8', 'Unnamed: 9', 
                   'Stock tracking model (ATS) : Opening Stock + Δ - Reserve', 'Fabric Name']
# Step 3: Apply the standardized header cleaning function
fabric_daily_log_df = clean_headers(fabric_daily_log_df)
# Step 4: Print and inspect the cleaned column names list
print(f"dataset : 'fabric_daily_log_df'")
print(fabric_daily_log_df.columns.tolist())

dataset : 'fabric_daily_log_df'
['date', 'month', 'year', 'fabric_sku', 'status', 'action_type', 'qty_change']


In [6]:
# Step 1: Load the "Finished Clothes Daily Log" CSV file into a pandas DataFrame
finished_clothes_daily_log_df = pd.read_csv("../data/raw_data/Finished_Clothes_Daily_Log.csv")

# Step 2: Rename certain columns
finished_clothes_daily_log_df = finished_clothes_daily_log_df.rename(
    columns={ "Product Full Name": "Product name"})

# Step 3 List of columns to delete 
columns_to_drop = ['Key2','Combo_SKU_Custom', "Size",'Combo_VPN_Size',
                   'Combo_SKU_Custom', 'Combo_Product_SKU', "Model", "VPN", "Fabric"]

# Step 4: Apply the standardized header cleaning function
finished_clothes_daily_log_df = clean_headers(finished_clothes_daily_log_df)
# Step 5: Print and inspect the cleaned column names list
print(f"dataset : 'finished_clothes_daily_log_df'")
print(finished_clothes_daily_log_df.columns.tolist())

dataset : 'finished_clothes_daily_log_df'
['date', 'product_name', 'height(cm)', 'product_sku', 'stock_type', 'quality_type', 'storage_ref', 'action_type', 'qty']


In [7]:
# Step 1: Load the "Finished Clothes Inventory" CSV file into a pandas DataFrame
Finished_Clothes_Inventory_df = pd.read_csv("../data/raw_data/Finished_Clothes_Inventory.csv")
# Step 2 List of columns to delete 
columns_to_drop = ["Product Model", "Fabric Name"]
# Step 3: Apply the standardized header cleaning function
Finished_Clothes_Inventory_df = clean_headers(Finished_Clothes_Inventory_df)
# Step 4: Print and inspect the cleaned column names list
print(f"dataset : 'Finished_Clothes_Inventory_df'")
print(Finished_Clothes_Inventory_df.columns.tolist())

dataset : 'Finished_Clothes_Inventory_df'
['appartment', 'room', 'box', 'stock_category', 'product_quality', 'product_sku', 'product_type', 'product_name', 'size', 'height', 'oh']


In [8]:
# Step 1: Load the " Procurement Tracker" CSV file into a pandas DataFrame
Procurement_Tracker_df = pd.read_csv("../data/raw_data/Procurement_Tracker.csv")
# Step 2 List of columns to delete 
columns_to_drop = ['Vendor', 'PO Number','Fabric ID Number', 'Season','Image','Material (weave)', 
                   'Former name', 'Sides',  'number of bags or rolls', 'Estimated Ship Date','Meters Shipped',
                  'total due based on shipped', 'Tracking', 'Actual Ship Date', 'Actual Delivery Date', 
                  'Lead Time', 'Meters short', 'price', 'total due based on received', 'PAYMENT STATUS', 
                   'PAYMENT DATE', 'Meters Received', 'Actual Delivery Date.1', 'Added to Inventory IN?',
                   'Date Added to Inventory IN', 'Notes', 'Weight', "Bag Number",'Name']
# Step 3: Apply the standardized header cleaning function
Procurement_Tracker_df  = clean_headers(Procurement_Tracker_df)
# Step 4: Print and inspect the cleaned column names list
print(f"dataset : 'Procurement_Tracker_df'")
print(Procurement_Tracker_df.columns.tolist())

dataset : 'Procurement_Tracker_df'
['year', 'order_date', 'month', 'stock_or_sample', 'fabric_sku', 'composition_(fibers)', 'total_meters_orderered', 'shipping_method', 'expected_delivery_(eta)']


### II- *Database Anonymization*

In [9]:
# Function to extract base material identifiers from raw fabric SKUs
fabric_sku_mapping = {}
def transform_fabric_sku(sku):
    """
    Anonymizes fabric SKUs by isolating the primary material code (first segment before hyphen).
    Preserves raw material relationships across datasets via mapping.
    """
    # Step 1: Return pd.NA for missing or empty SKU values
    if pd.isna(sku) or str(sku).strip() == "":
        return pd.NA

    # Step 2: Split SKU string on hyphen to separate material attributes
    sku_str = str(sku).strip()
    fabric_sku = sku_str.split("-")

    # Step 3: Extract the primary prefix code
    if len(fabric_sku) > 1:
        part = fabric_sku[0]
    else:
        part = fabric_sku[0]

    # Step 4: Record in global dictionary and return the anonymized fabric code
    fabric_sku_mapping[sku] = part
    return fabric_sku_mapping[sku]

In [10]:
# Extract primary material codes to anonymize raw fabric SKUs across list of databases
databases = {
    "fabric_daily_log": fabric_daily_log_df,
    "procurement_tracker": Procurement_Tracker_df,
    "fabric_stock": fabric_stock_df }

for name, df in databases.items():
    if "fabric_sku" in df.columns:
        df["fabric_sku"] = df["fabric_sku"].apply(transform_fabric_sku)

In [11]:
# Add anonymized product model column to finished_clothes_daily_log  dataset 
finished_clothes_daily_log_df["product_model"] = finished_clothes_daily_log_df[
    "product_sku"].apply(
    lambda x: (
        f"{str(x).split('-')[0]}1"
        if len(str(x).split("-")) >= 1 and str(x).split("-")[0] != ""
        else ""))

In [12]:
# Add anonymized product model column to Finished_Clothes_Inventory dataset
Finished_Clothes_Inventory_df["product_model"] = Finished_Clothes_Inventory_df[
    "product_sku"].apply(
    lambda x: (
        f"{str(x).split('-')[0]}1"
        if len(str(x).split("-")) >= 1 and str(x).split("-")[0] != ""
        else ""))

In [13]:
fabric_daily_log_df.head(3)

,date,month,year,fabric_sku,status,action_type,qty_change
0,1/2/2026,January,2026,ROC08,Active,Web orders,-8
1,1/2/2026,January,2026,COC08,hold,Web orders,-5.1
2,1/2/2026,January,2026,CHI08,Active,Web orders,-4


In [14]:
fabric_stock_df.head(3)

,fabric_sku,print,color,composition,weave,season,ats
0,ABS08,Abstract,Noir,100% Polyester Jacquard,Jacquard,Fall 2022,5.40
1,ABS08,Abstract,Rose,100% Polyester Jacquard,Jacquard,Spring 2023,113.02
2,ABS12,Abstract,Rose,100% Polyester Chiffon,Chiffon,Spring 2023,66.05


In [15]:
Procurement_Tracker_df.head(3)

,year,order_date,month,stock_or_sample,fabric_sku,composition_(fibers),total_meters_orderered,shipping_method,expected_delivery_(eta)
0,2026,"Sunday, 09/13",September,Stock,ULA12,Sample Fabric - Input,60.0,SEA freight,"Dec 22, 2026"
1,2026,"Sunday, 09/13",September,Stock,DOR08,100% Polyester Jacquard,80.0,SEA freight,"Dec 22, 2026"
2,2026,"Sunday, 09/13",September,Stock,ZIM08,100% Polyester Jacquard,100.0,SEA freight,"Dec 22, 2026"


In [16]:
# Add anonymized Fabric SKU column to finished_clothes_daily_log dataset  
finished_clothes_daily_log_df["fabric_sku"] = finished_clothes_daily_log_df[
    "product_sku"].apply(
    lambda x: (
        f"{str(x).split('-')[1]}"
        if len(str(x).split("-")) >= 1 and str(x).split("-")[1] != ""
        else ""))

In [17]:
# Add anonymized Fabric SKU column to Finished_Clothes_Inventory dataset  
Finished_Clothes_Inventory_df["fabric_sku"] = Finished_Clothes_Inventory_df[
    "product_sku"].apply(
    lambda x: (
        f"{str(x).split('-')[1]}"
        if len(str(x).split("-")) >= 1 and str(x).split("-")[1] != ""
        else ""))

In [18]:
# Global dictionary to store original-to-cleaned name mappings
product_mapping = {}
def product_name(name):
    """Parses and cleans raw product names into a standardized 'Category - Variant' format.

    Extracts the last word before and after the first hyphen ('-') and updates a
    global mapping dictionary.
    """
    # 1. Handle missing, null, or whitespace-only inputs
    if pd.isna(name) or str(name).strip() == "":
        return pd.NA
    # 2. Convert input to string and split by hyphen
    str_name = str(name).strip().split("-")
    # 3. Handle cases where no hyphen is present in the product name
    if len(str_name) <= 1:
        product_name = str_name[0].strip()
    # 4. Handle hyphenated names (e.g., 'Sustainable Leer Pants - Solid Cream')
    else:
        # Split text before and after the hyphen into word lists
        part0 = str_name[0].split()
        part1 = str_name[1].split()
        # Extract the last word from each side (safely handling empty lists)
        part0 = part0[-1].strip() if part0 else ""
        part1 = part1[-1].strip() if part1 else ""
        # Format into standardized string
        product_name = f"{part0} - {part1}"
    # 5. Record the mapping entry and return the cleaned value
    product_mapping[name] = product_name
    return product_mapping[name]

In [19]:
if 'product_name' in finished_clothes_daily_log_df.columns : 
    finished_clothes_daily_log_df['product_name'] = finished_clothes_daily_log_df['product_name'].apply(product_name)

if 'product_name' in Finished_Clothes_Inventory_df.columns : 
    Finished_Clothes_Inventory_df['product_name'] = Finished_Clothes_Inventory_df['product_name'].apply(product_name)

In [20]:
# Global dictionary to store original-to-cleaned SKU mappings
product_sku_mapping = {}
def product_sku(sku):
    """Normalizes raw product SKUs by removing middle segments and updates a
    global mapping dictionary.
    """
    # 1. Handle missing, null, or whitespace-only inputs
    if pd.isna(sku) or str(sku).strip() == "":
        return pd.NA

    # 2. Convert input to string, remove extra spaces, and split by hyphen
    sku_parts = str(sku).strip().split("-")
    # 3. Process SKU based on the number of hyphen-separated segments
    if len(sku_parts) < 3:
        # Rejoin original parts if there are fewer than 3 segments
        clean_product_sku = "-".join(sku_parts)
    else:
        # Remove middle segments (indexes 1 and 2)
        del sku_parts[1:3]
        clean_product_sku = "-".join(sku_parts)
    # 4. Record the mapping entry and return the cleaned SKU value
    product_sku_mapping[sku] = clean_product_sku
    return product_sku_mapping[sku]

In [21]:
 if 'product_sku' in finished_clothes_daily_log_df.columns : 
    finished_clothes_daily_log_df['product_sku'] = finished_clothes_daily_log_df['product_sku'].apply(product_sku)

if 'product_sku' in Finished_Clothes_Inventory_df.columns : 
    Finished_Clothes_Inventory_df['product_sku'] = Finished_Clothes_Inventory_df['product_sku'].apply(product_sku)

In [22]:
finished_clothes_daily_log_df.head(2)

,date,product_name,height(cm),product_sku,stock_type,quality_type,storage_ref,action_type,qty,product_model,fabric_sku
0,1/22/2026,Dress - Teal,173.0,DMAY0-2XL,Finished product,Second,APT02-CHB01-2ND-BOX-3,Storage,1.0,DMAY01,SOL14
1,1/22/2026,Dress - Noir,173.0,DMAY0-2XL,Finished product,Second,APT02-CHB01-2ND-BOX-3,Storage,1.0,DMAY01,SOL14


In [23]:
Finished_Clothes_Inventory_df.head(2)

,appartment,room,box,stock_category,product_quality,product_sku,product_type,product_name,size,height,oh,product_model,fabric_sku
0,APT02,CHB01,2ND-BOX-3,Finished product,Second,DMAY0-2XL,Dress,Dress - Noir,2XL,173.0,6,DMAY01,SOL14
1,APT02,CHB01,2ND-BOX-3,Finished product,Second,DMAY0-3XL,Dress,Dress - Noir,3XL,173.0,2,DMAY01,SOL14


### III  Exploratory Data Analysis

In [24]:
#Procurement_Tracker_df

###  Finished lothes Inventory Dataframe

In [25]:
Finished_Clothes_Inventory_df.shape

(1550, 13)

In [26]:
Finished_Clothes_Inventory_df.head(3)

,appartment,room,box,stock_category,product_quality,product_sku,product_type,product_name,size,height,oh,product_model,fabric_sku
0,APT02,CHB01,2ND-BOX-3,Finished product,Second,DMAY0-2XL,Dress,Dress - Noir,2XL,173.0,6,DMAY01,SOL14
1,APT02,CHB01,2ND-BOX-3,Finished product,Second,DMAY0-3XL,Dress,Dress - Noir,3XL,173.0,2,DMAY01,SOL14
2,APT02,CHB01,2ND-BOX-1,Finished product,Second,JUMYO-S,Jumpsuit,Jumpsuit - vert,S,173.0,9,JUMYO1,SAL08


In [27]:
Finished_Clothes_Inventory_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1550 entries, 0 to 1549
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appartment       1550 non-null   object 
 1   room             1550 non-null   object 
 2   box              1550 non-null   object 
 3   stock_category   1550 non-null   object 
 4   product_quality  1550 non-null   object 
 5   product_sku      1550 non-null   object 
 6   product_type     1550 non-null   object 
 7   product_name     1550 non-null   object 
 8   size             1549 non-null   object 
 9   height           1401 non-null   float64
 10  oh               1550 non-null   int64  
 11  product_model    1550 non-null   object 
 12  fabric_sku       1550 non-null   object 
dtypes: float64(1), int64(1), object(11)
memory usage: 157.6+ KB


In [28]:
#Checking Numeric Variables function
def inspect_numeric(df):
    # En une seule ligne : attrape tous les int, float, etc.
    num_cols = df.select_dtypes(include=["number"]).columns

    return pd.DataFrame(
        {
            "data_type": df[num_cols].dtypes,
            "missing_count": df[num_cols].isna().sum(),
            "missing_pct (%)": (df[num_cols].isna().mean() * 100).round(2),
            "zero_count": (df[num_cols] == 0).sum(),
            "min": df[num_cols].min(),
            "median": df[num_cols].median(),
            "mean": df[num_cols].mean().round(2),
            "max": df[num_cols].max(),
        }
    )

In [29]:
inspect_numeric(Finished_Clothes_Inventory_df)

,data_type,missing_count,missing_pct (%),zero_count,min,median,mean,max
height,float64,149,9.61,0,150.0,173.0,172.82,186.0
oh,int64,0,0.00,0,1.0,2.0,3.38,52.0


In [30]:
#Checking Categorical Variables function
def inspect_categorical(df):
    cat_cols = df.select_dtypes(exclude=["number"]).columns

    return pd.DataFrame(
        {
            "data_type": df[cat_cols].dtypes,
            "missing_count": df[cat_cols].isna().sum(),
            "missing_pct (%)": (df[cat_cols].isna().mean() * 100).round(2),
            "unique_count": df[cat_cols].nunique(),
            "sample_values": [
                df[col].dropna().unique()[:3].tolist() for col in cat_cols
            ],
        }
    )

In [31]:
inspect_categorical(Finished_Clothes_Inventory_df)

,data_type,missing_count,missing_pct (%),unique_count,sample_values
appartment,object,0,0.00,2,"[APT02, APT01]"
room,object,0,0.00,2,"[CHB01, CHA01]"
box,object,0,0.00,109,"[2ND-BOX-3, 2ND-BOX-1, 2ND-BOX-53]"
stock_category,object,0,0.00,1,[Finished product]
product_quality,object,0,0.00,2,"[Second, First]"
product_sku,object,0,0.00,348,"[DMAY0-2XL, DMAY0-3XL, JUMYO-S]"
product_type,object,0,0.00,13,"[Dress, Jumpsuit, Pants]"
product_name,object,0,0.00,162,"[Dress - Noir, Jumpsuit - vert, Jumpsuit - Rust]"
size,object,1,0.06,10,"[2XL, 3XL, S]"
product_model,object,0,0.00,85,"[DMAY01, JUMYO1, JUMY01]"


In [32]:
# 1. Define the relative path to the clean data directory
clean_data = Path("../Data/clean_data")
# 2. Ensure the directory exists on the disk (creates it automatically if needed)
clean_data.mkdir(parents=True, exist_ok=True)
# 3. Export the clean CSV file 
Finished_Clothes_Inventory_df.to_csv(clean_data / "Finished_Clothes_Inventory.csv", index=False)
print("clean datasets successfully exported to Data/clean_data")

clean datasets successfully exported to Data/clean_data


### finished clothes daily log Dataframe

In [33]:
finished_clothes_daily_log_df.shape

(2945, 11)

In [34]:
finished_clothes_daily_log_df.head()

,date,product_name,height(cm),product_sku,stock_type,quality_type,storage_ref,action_type,qty,product_model,fabric_sku
0,1/22/2026,Dress - Teal,173.0,DMAY0-2XL,Finished product,Second,APT02-CHB01-2ND-BOX-3,Storage,1.0,DMAY01,SOL14
1,1/22/2026,Dress - Noir,173.0,DMAY0-2XL,Finished product,Second,APT02-CHB01-2ND-BOX-3,Storage,1.0,DMAY01,SOL14
2,1/22/2026,Dress - Noir,173.0,DMAY0-2XL,Finished product,Second,APT02-CHB01-2ND-BOX-3,Storage,1.0,DMAY01,SOL14
3,1/22/2026,Dress - Noir,173.0,DMAY0-2XL,Finished product,Second,APT02-CHB01-2ND-BOX-3,Storage,1.0,DMAY01,SOL14
4,1/22/2026,Dress - Noir,173.0,DMAY0-2XL,Finished product,Second,APT02-CHB01-2ND-BOX-3,Storage,1.0,DMAY01,SOL14


In [35]:
finished_clothes_daily_log_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2945 entries, 0 to 2944
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           2945 non-null   object 
 1   product_name   2945 non-null   object 
 2   height(cm)     2692 non-null   float64
 3   product_sku    2945 non-null   object 
 4   stock_type     2945 non-null   object 
 5   quality_type   2945 non-null   object 
 6   storage_ref    2945 non-null   object 
 7   action_type    2945 non-null   object 
 8   qty            2934 non-null   float64
 9   product_model  2945 non-null   object 
 10  fabric_sku     2945 non-null   object 
dtypes: float64(2), object(9)
memory usage: 253.2+ KB


In [36]:
inspect_numeric(finished_clothes_daily_log_df)

,data_type,missing_count,missing_pct (%),zero_count,min,median,mean,max
height(cm),float64,253,8.59,0,150.0,173.0,172.93,190.0
qty,float64,11,0.37,38,-23.0,1.0,1.78,53.0


In [37]:
finished_clothes_daily_log_df = finished_clothes_daily_log_df.dropna(subset = ["qty"])

In [38]:
inspect_numeric(finished_clothes_daily_log_df)

,data_type,missing_count,missing_pct (%),zero_count,min,median,mean,max
height(cm),float64,253,8.62,0,150.0,173.0,172.93,190.0
qty,float64,0,0.00,38,-23.0,1.0,1.78,53.0


In [39]:
inspect_categorical(finished_clothes_daily_log_df)

,data_type,missing_count,missing_pct (%),unique_count,sample_values
date,object,0,0.0,129,"[1/22/2026, 1/23/2026, 1/26/2026]"
product_name,object,0,0.0,190,"[Dress - Teal, Dress - Noir, Jumpsuit - vert]"
product_sku,object,0,0.0,435,"[DMAY0-2XL, DMAY0-3XL, DMAY0-XS]"
stock_type,object,0,0.0,1,[Finished product]
quality_type,object,0,0.0,2,"[Second, First]"
storage_ref,object,0,0.0,112,"[APT02-CHB01-2ND-BOX-3, APT02-CHB01-2ND-BOX-1,..."
action_type,object,0,0.0,2,"[Storage, Destocking]"
product_model,object,0,0.0,106,"[DMAY01, JUMYO1, JUMY01]"
fabric_sku,object,0,0.0,120,"[SOL14, SAL08, TUK08]"


In [40]:
# Convert raw date strings (US format M/D/YYYY) to Pandas datetime objects, turning invalid inputs into NaT (Not a Time)
finished_clothes_daily_log_df["date"] = pd.to_datetime(
    finished_clothes_daily_log_df["date"], format="%m/%d/%Y", errors="coerce")

# Format valid datetime objects into standard ISO strings (YYYY-MM-DD) for SQLite and Looker Studio compatibility
finished_clothes_daily_log_df["date"] = finished_clothes_daily_log_df["date"].dt.strftime("%Y-%m-%d")

In [41]:
# 1. Define the relative path to the clean data directory
clean_data = Path("../Data/clean_data")
# 2. Ensure the directory exists on the disk (creates it automatically if needed)
clean_data.mkdir(parents=True, exist_ok=True)
# 3. Export the clean CSV file 
finished_clothes_daily_log_df.to_csv(clean_data / "finished_clothes_daily_log.csv", index=False)
print("clean datasets successfully exported to Data/clean_data")

clean datasets successfully exported to Data/clean_data


###  fabric stock Dataframe

In [42]:
fabric_stock_df.shape

(330, 7)

In [43]:
fabric_stock_df.head()

,fabric_sku,print,color,composition,weave,season,ats
0,ABS08,Abstract,Noir,100% Polyester Jacquard,Jacquard,Fall 2022,5.40
1,ABS08,Abstract,Rose,100% Polyester Jacquard,Jacquard,Spring 2023,113.02
2,ABS12,Abstract,Rose,100% Polyester Chiffon,Chiffon,Spring 2023,66.05
3,ALI08,Alia,Noir,100% Polyester Jacquard,Jacquard,Resort 2024,0.31
4,ANA08,Ananas,Black,100% Polyester Jacquard,Jacquard,Resort 2021,0.00


In [44]:
inspect_numeric(fabric_stock_df)

,data_type,missing_count,missing_pct (%),zero_count,min,median,mean,max
ats,float64,3,0.91,181,0.0,0.0,43.73,2052.41


In [45]:
fabric_stock_df = fabric_stock_df.dropna(subset = ["ats"])

In [46]:
inspect_numeric(fabric_stock_df)

,data_type,missing_count,missing_pct (%),zero_count,min,median,mean,max
ats,float64,0,0.0,181,0.0,0.0,43.73,2052.41


In [47]:
inspect_categorical(fabric_stock_df)

,data_type,missing_count,missing_pct (%),unique_count,sample_values
fabric_sku,object,0,0.00,219,"[ABS08, ABS12, ALI08]"
print,object,0,0.00,124,"[Abstract, Alia, Ananas]"
color,object,0,0.00,37,"[Noir, Rose, Black]"
composition,object,0,0.00,31,"[100% Polyester Jacquard, 100% Polyester Chiff..."
weave,object,127,38.84,8,"[Jacquard, Chiffon, Plain]"
season,object,128,39.14,19,"[Fall 2022, Spring 2023, Resort 2024]"


In [48]:
# 1. Define the relative path to the clean data directory
clean_data = Path("../Data/clean_data")
# 2. Ensure the directory exists on the disk (creates it automatically if needed)
clean_data.mkdir(parents=True, exist_ok=True)
# 3. Export the clean CSV file 
fabric_stock_df.to_csv(clean_data / "fabric_stock.csv", index=False)
print("clean datasets successfully exported to Data/clean_data")

clean datasets successfully exported to Data/clean_data


### fabric daily log Dataframe

In [49]:
fabric_daily_log_df.shape

(2474, 7)

In [50]:
fabric_daily_log_df.head()

,date,month,year,fabric_sku,status,action_type,qty_change
0,1/2/2026,January,2026,ROC08,Active,Web orders,-8
1,1/2/2026,January,2026,COC08,hold,Web orders,-5.1
2,1/2/2026,January,2026,CHI08,Active,Web orders,-4
3,1/2/2026,January,2026,ISL12,Discontinued,Web orders,-5
4,1/2/2026,January,2026,SIE08,Active,Web orders,-7


In [51]:
fabric_daily_log_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2474 entries, 0 to 2473
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         2474 non-null   object
 1   month        2474 non-null   object
 2   year         2474 non-null   int64 
 3   fabric_sku   2474 non-null   object
 4   status       2467 non-null   object
 5   action_type  2474 non-null   object
 6   qty_change   2474 non-null   object
dtypes: int64(1), object(6)
memory usage: 135.4+ KB


In [52]:
# 1. Safely convert to numeric (forces any remaining bad text to NaN)
fabric_daily_log_df["qty_change"] = pd.to_numeric(
    fabric_daily_log_df["qty_change"], errors="coerce")

# 2. Fill missing values with 0 and convert to float
fabric_daily_log_df["qty_change"] = (
    fabric_daily_log_df["qty_change"].fillna(0).astype(float)
)

# 3. Invert the quantity change sign
fabric_daily_log_df["qty_change"] = fabric_daily_log_df["qty_change"] * -1

In [53]:
fabric_daily_log_df.head()

,date,month,year,fabric_sku,status,action_type,qty_change
0,1/2/2026,January,2026,ROC08,Active,Web orders,8.0
1,1/2/2026,January,2026,COC08,hold,Web orders,5.1
2,1/2/2026,January,2026,CHI08,Active,Web orders,4.0
3,1/2/2026,January,2026,ISL12,Discontinued,Web orders,5.0
4,1/2/2026,January,2026,SIE08,Active,Web orders,7.0


In [54]:
inspect_numeric(fabric_daily_log_df)

,data_type,missing_count,missing_pct (%),zero_count,min,median,mean,max
year,int64,0,0.0,0,2026.0,2026.0,2026.00,2026.0
qty_change,float64,0,0.0,5,-570.7,3.7,2.11,201.0


In [55]:
inspect_categorical(fabric_daily_log_df)

,data_type,missing_count,missing_pct (%),unique_count,sample_values
date,object,0,0.00,169,"[1/2/2026, 1/5/2026, 1/7/2026]"
month,object,0,0.00,9,"[January, February, March]"
fabric_sku,object,0,0.00,116,"[ROC08, COC08, CHI08]"
status,object,7,0.28,6,"[Active, hold, Discontinued]"
action_type,object,0,0.00,6,"[Web orders, Stock received, Wholesales Orders]"


In [56]:
# Convert raw date strings (US format M/D/YYYY) to Pandas datetime objects, turning invalid inputs into NaT (Not a Time)
fabric_daily_log_df["date"] = pd.to_datetime(
    fabric_daily_log_df["date"], format="%m/%d/%Y", errors="coerce")

# Format valid datetime objects into standard ISO strings (YYYY-MM-DD) for SQLite and Looker Studio compatibility
fabric_daily_log_df["date"] = fabric_daily_log_df["date"].dt.strftime("%Y-%m-%d")

In [57]:
# 1. Define the relative path to the clean data directory
clean_data = Path("../Data/clean_data")
# 2. Ensure the directory exists on the disk (creates it automatically if needed)
clean_data.mkdir(parents=True, exist_ok=True)
# 3. Export the clean CSV file 
fabric_daily_log_df.to_csv(clean_data / "fabric_daily_log.csv", index=False)
print("clean datasets successfully exported to Data/clean_data")

clean datasets successfully exported to Data/clean_data


### Procurement Tracker DataFrame 

In [58]:
Procurement_Tracker_df.shape

(43, 9)

In [59]:
Procurement_Tracker_df.head()

,year,order_date,month,stock_or_sample,fabric_sku,composition_(fibers),total_meters_orderered,shipping_method,expected_delivery_(eta)
0,2026,"Sunday, 09/13",September,Stock,ULA12,Sample Fabric - Input,60.0,SEA freight,"Dec 22, 2026"
1,2026,"Sunday, 09/13",September,Stock,DOR08,100% Polyester Jacquard,80.0,SEA freight,"Dec 22, 2026"
2,2026,"Sunday, 09/13",September,Stock,ZIM08,100% Polyester Jacquard,100.0,SEA freight,"Dec 22, 2026"
3,2026,"Sunday, 09/13",September,Stock,ZIM08,100% Polyester Jacquard,100.0,AIR freight,"Oct 13, 2026"
4,2026,"Sunday, 09/13",September,Stock,AYE29,"96% Polyester, 4% Spandex (Ayel Weave)",50.0,SEA freight,"Dec 22, 2026"


In [60]:
Procurement_Tracker_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   year                     43 non-null     int64  
 1   order_date               43 non-null     object 
 2   month                    43 non-null     object 
 3   stock_or_sample          43 non-null     object 
 4   fabric_sku               43 non-null     object 
 5   composition_(fibers)     43 non-null     object 
 6   total_meters_orderered   43 non-null     float64
 7   shipping_method          43 non-null     object 
 8   expected_delivery_(eta)  43 non-null     object 
dtypes: float64(1), int64(1), object(7)
memory usage: 3.2+ KB


In [61]:
inspect_numeric(Procurement_Tracker_df)

,data_type,missing_count,missing_pct (%),zero_count,min,median,mean,max
year,int64,0,0.0,0,2026.00,2026.0,2026.00,2026.0
total_meters_orderered,float64,0,0.0,0,19.43,132.0,134.94,300.0


In [62]:
inspect_categorical(Procurement_Tracker_df)

,data_type,missing_count,missing_pct (%),unique_count,sample_values
order_date,object,0,0.0,13,"[Sunday, 09/13, Wednesday, 09/09, Monday, 09/07]"
month,object,0,0.0,5,"[September, August, July]"
stock_or_sample,object,0,0.0,2,"[Stock, Sample]"
fabric_sku,object,0,0.0,31,"[ULA12, DOR08, ZIM08]"
composition_(fibers),object,0,0.0,7,"[Sample Fabric - Input, 100% Polyester Jacquar..."
shipping_method,object,0,0.0,2,"[SEA freight, AIR freight]"
expected_delivery_(eta),object,0,0.0,10,"[Dec 22, 2026, Oct 13, 2026, Oct 5, 2026]"


In [63]:
# 1. Extract only the MM/DD part (ignores "Sunday, " or any leading/trailing spaces)
mm_dd_series = Procurement_Tracker_df["order_date"].str.extract(r"(\d{1,2}/\d{1,2})")[0]

# 2. Clean the year column to remove potential decimals (e.g., 2026.0 -> 2026)
year_series = (
    Procurement_Tracker_df["year"]
    .fillna(0)
    .astype(int)
    .astype(str)
    .replace("0", ""))

# 3. Combine into MM/DD/YYYY format
full_date_series = mm_dd_series + "/" + year_series

# 4. Parse using standard MM/DD/YYYY format
Procurement_Tracker_df["order_date"] = pd.to_datetime(
    full_date_series, format="%m/%d/%Y", errors="coerce")

# 5. Format to standard ISO string (YYYY-MM-DD) for SQLite and Looker Studio
Procurement_Tracker_df["order_date"] = Procurement_Tracker_df["order_date"].dt.strftime(
    "%Y-%m-%d")

In [68]:
# Convert raw string dates (e.g., "Dec 22, 2026") into Pandas datetime objects
# %b = Abbreviated month name (Dec, Jan, Feb...), %d = Day of the month, %Y = 4-digit Year
Procurement_Tracker_df["expected_delivery_(eta)"] = pd.to_datetime(
    Procurement_Tracker_df["expected_delivery_(eta)"], format="%b %d, %Y", errors="coerce"
)

# Format valid datetime objects into standard ISO strings (YYYY-MM-DD) for SQLite & Looker Studio
Procurement_Tracker_df["expected_delivery_(eta)"] = Procurement_Tracker_df["expected_delivery_(eta)"].dt.strftime(
    "%Y-%m-%d"
)

In [69]:
Procurement_Tracker_df["expected_delivery_(eta)"].head()

0    2026-12-22
1    2026-12-22
2    2026-12-22
3    2026-10-13
4    2026-12-22
Name: expected_delivery_(eta), dtype: object

In [70]:
Procurement_Tracker_df.head()

,year,order_date,month,stock_or_sample,fabric_sku,composition_(fibers),total_meters_orderered,shipping_method,expected_delivery_(eta)
0,2026,2026-09-13,September,Stock,ULA12,Sample Fabric - Input,60.0,SEA freight,2026-12-22
1,2026,2026-09-13,September,Stock,DOR08,100% Polyester Jacquard,80.0,SEA freight,2026-12-22
2,2026,2026-09-13,September,Stock,ZIM08,100% Polyester Jacquard,100.0,SEA freight,2026-12-22
3,2026,2026-09-13,September,Stock,ZIM08,100% Polyester Jacquard,100.0,AIR freight,2026-10-13
4,2026,2026-09-13,September,Stock,AYE29,"96% Polyester, 4% Spandex (Ayel Weave)",50.0,SEA freight,2026-12-22


In [71]:
# 1. Define the relative path to the clean data directory
clean_data = Path("../Data/clean_data")
# 2. Ensure the directory exists on the disk (creates it automatically if needed)
clean_data.mkdir(parents=True, exist_ok=True)
# 3. Export the clean CSV file 
Procurement_Tracker_df.to_csv(clean_data / "Procurement_Tracker.csv", index=False)
print("clean datasets successfully exported to Data/clean_data")

clean datasets successfully exported to Data/clean_data
